<table>
  <tr>
    <td><div align="left"><font size="30">Modeling a perspective camera</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png?raw=1" width="200" height="158" style="height:auto"></td>
  </tr>
</table>

(c) Peter Corke 2024

Robotics, Vision & Control: Python, see section 13.1.2

## Configuring the Jupyter environment
We need to import some packages to help us with linear algebra (`numpy`), graphics (`matplotlib`), and machine vision (`machinevisiontoolbox`).
If you're running locally you need to have these packages installed.  If you're running on CoLab we have to first install machinevisiontoolbox which is not preinstalled, this will be a bit slow.

In [ ]:
# MVTB_BOOTSTRAP_CELL: installs the package, which takes a short time, so wait for the line "Running ..." below.  Generated by sync_bootstrap.py from _mvtb_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'machinevision-toolbox-python'
IMPORT_NAME = 'machinevisiontoolbox'
GIT_URL: str | None = 'https://github.com/petercorke/machinevision-toolbox-python'
JL_PACKAGES = ['numpy', 'scipy', 'matplotlib', 'pillow', 'typing_extensions', 'opencv-python', 'ansitable', 'mvtb-data', 'tqdm', 'requests', 'ipywidgets', 'spatialmath-python', 'pgraph-python']
JL_NO_DEPS: list[str] = []
JL_PACKAGE_NO_DEPS = True
JL_EXTRAS: dict[str, list[str]] = {}
COLAB_PACKAGES: list[str] = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE, deps=not JL_PACKAGE_NO_DEPS)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}, this takes a short time...', flush=True)
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import mpl_toolkits.mplot3d.art3d as art3d
import ipywidgets as widgets
%matplotlib inline

***

The sort of camera that we have in our eye and in our mobile phone performs a mapping from the 3D world to a 2D image. The particular mapping is called a perspective projection, and the underlying mathematics is very simple.

Two models are generally used:

1.  The "pinhole camera" model, which produces an inverted image, using a pinhole approximation to a lens.
2.  The central projection model, widely used in the computer vision literature, which creates a non-inverted image.

For a camera at the origin looking out along the z-axis (its optical axis) and a point in the world at $P = (X, Y, Z)$ the 2D projection on the image plane will be $p=(x,y)$ where
\begin{equation}
x = \frac{fX}{Z}, y = \frac{fY}{Z}
\end{equation}
and $f$ is the focal length of the lense.

The simple animation below lets you adjust the coordinates $(X, Y, Z)$ and that alters the projection ray, from
the origin through that point. Where that ray pierces the imageplane at $Z=f$ is the image-plane projection of that world point.

Have a play.  Convince yourself that an infinite number of points in the world have the same image-plane projection.

In [ ]:
verts = np.array([
    [-5, -5],
    [-5,  5],
    [ 5,  5],
    [ 5, -5] ])

@widgets.interact
def animate( X = widgets.FloatSlider(value=1, description='X:', min=-4, max=4),
             Y = widgets.FloatSlider(value=1, description='Y:', min=-4, max=4),
             Z = widgets.FloatSlider(value=2, description='Z:', min=1, max=5),
             f = widgets.FloatSlider(value=1, description='f', min=0.1, max=1.5)):
    fig = plt.figure(figsize=(10, 10))
    ax = plt.axes(projection='3d')
    p = plt.Polygon(verts, alpha=0.7, color="lightyellow")

    ax.add_patch(p)
    art3d.pathpatch_2d_to_3d(p, z=f, zdir="z")

    ax.set_xlim3d(-5, 5)
    ax.set_ylim3d(-5, 5)
    ax.set_zlim3d(0, 5)
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    P = ax.scatter3D(X, Y, Z, 'bo')
    P.set_sizes([40])
    ax.quiver(0, 0, 0, X, Y, Z, arrow_length_ratio=0.1)
    x = X * f / Z
    y = Y * f / Z
    p = ax.scatter3D(x, y, f, 'ko')
    p.set_sizes([20])

    ax.text(X, Y, Z, "   P")
    ax.text(x, y, f,  "  p = (%.3f, %.3f)" % (x, y))
    plt.show()
    
    # uv_coords.value = "Image-plane coordinate:   (%.3f, %.3f)" % (x, y)
